# Agreement Matrices Under Prompt Variations

Pairwise Cohen kappa agreement matrices across models for Prompt A, Prompt B, and Prompt A without the letter token. Confidence intervals are estimated with row-level bootstrap resampling.


In [ ]:
import os
import pandas as pd
import json
from os.path import join
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import cohen_kappa_score
from collections import defaultdict
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import scipy.stats as st
import matplotlib.cm as cm
import matplotlib.colorbar as colorbar
from statsmodels.stats.contingency_tables import cochrans_q, mcnemar, Table
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.proportion import proportion_confint
import plotly.graph_objects as go


import itertools

In [ ]:
# ESAP questions with images
q_w_images = [3, 4, 6, 32, 35, 54, 68, 88]

cases = np.delete(np.arange(100), [0] + q_w_images) # valid questions
n_cases = cases.size

# Available options
options = ['A', 'B', 'C', 'D', 'E']
# even for the experiments without letter, manual annotator saved the option using letter.

path_to_results = "./../results"

model_file_name_dict = {'huatuo-o1': 'HuatuoGPT-o1',
                        'diabetica-o1': 'Diabetica-o1',
                        'diabetica-7B': 'Diabetica-7B',
                        'meditron3-8B': 'Meditron3-8B'}

In [ ]:
def evaluate_response_distrib(filepath):
    """ 
    Given the excel file, this function parses and standardize the file entries.
    Returns a counter in the form of a dictionary.
    """
    df_ = pd.read_excel(filepath, header=None, index_col=0)
    
    # everything capital
    df_ = df_.map(lambda x: x.upper() if isinstance(x, str) else x) 
    
    # remove NaN values - this remove extra-columns
    df_ = df_.dropna(axis=1, how='all') 
    if not np.array_equal(df_.index, cases): # we keep all 91 cases
        raise ValueError("We are removing a valid case by mistake! Check Excel file")

    # unique annotations
    # print(df_.values)
    print(np.unique(df_.values))
    unique_elements = np.unique(df_.values) 

    # multiple options selected
    # are indicated as option1-option2: there is a dash
    bm_multiple = ["-" in v_ for v_ in unique_elements] 
    
    bm_hall_none = [True if (v_ == "HALL" or v_ == 'NONE') # hallucination or no response
                    else False for v_ in unique_elements]
    
    bm_others = ~np.logical_or(bm_multiple, bm_hall_none) # correct answers / check (temporary)
    
    idx_multiple = np.where(bm_multiple)[0] # indexes
    idx_hall_none = np.where(bm_hall_none)[0]
        
    dct_counter = {}
    all_ = 0 # to make sure we did not miss any response (91 or multiple for 10 rep)

    # we are creating a new label for multi and hall/none
    if len(idx_multiple) > 0:
        count = 0
        for v_ in unique_elements[idx_multiple]:
            count += (df_.values == v_).sum()
        all_ = count
        dct_counter["MULTI"] = count
            
    if len(idx_hall_none) > 0:
        count = 0
        for v_ in unique_elements[idx_hall_none]:
            count += (df_.values == v_).sum()
        all_ += count
        dct_counter["HALL / NONE"] = count
       
    # we are using the old label for everything else
    for element in unique_elements[bm_others]:
        count = (df_.values == element).sum()
        dct_counter[element] = count
        all_ += count
            
    list_add = ["CHECK", "MULTI", "HALL / NONE", "A", "B", "C", "D", "E"]
    for el_ in list_add:
        if el_ not in dct_counter.keys():
            dct_counter[el_] = 0
    dct_counter["TOTAL"] = all_
        
    return dct_counter

In [ ]:
model_paths_token = {'meditron3-8B-prompt1': 'meditron3-8B_promptID_001_output_4.xlsx',
                     'huatuo-o1-prompt1': 'huatuo-o1_promptID_001_output_9.xlsx',
                     'huatuo-o1-prompt2': 'huatuo-o1_promptID_002_output_11.xlsx',
                     'diabetica-o1-prompt1': 'diabetica-o1_promptID_001_output_2.xlsx',
                     'diabetica-o1-prompt2': 'diabetica-o1_promptID_002_output_3.xlsx',
                     'diabetica-7B-prompt1': 'diabetica-7B_promptID_001_output_3.xlsx',
                     'diabetica-7B-prompt2': 'diabetica-7B_promptID_002_output_4.xlsx',
                     'medfound7B-prompt1': 'medfound7B_promptID_001_output_2.xlsx',
                     'medfound7B-prompt2': 'medfound7B_promptID_002_output_3.xlsx',
                     'clinical-chatgpt-prompt1': 'clinical-chatgpt_promptID_001_output_2.xlsx',
                     'clinical-chatgpt-prompt2': 'clinical-chatgpt_promptID_002_output_3.xlsx',
                     'bloom-7b-prompt1': 'bloom-7B_promptID_001_output_0.xlsx',
                     'bloom-7b-prompt2': 'bloom-7B_promptID_002_output_1.xlsx',
                     'llama31-prompt1': 'llama31_promptID_001_output_6.xlsx',
                     'llama31-prompt2': 'llama31_promptID_002_output_7.xlsx',
                     'qwen2-7b-prompt1': 'qwen2-7b_promptID_001_output_0.xlsx',
                     'qwen2-7b-prompt2': 'qwen2-7b_promptID_002_output_1.xlsx',
                     'huatuo-o1-lms-prompt1': 'huatuo-o1-lms-prompt001.xlsx',
                     'gpt-oss-20b-prompt1': 'gpt-oss-20b-lms-prompt001.xlsx'
                    }

model_paths_no_token = {'huatuo-o1': 'huatuo-o1_promptID_001_output_47.xlsx',
                        'diabetica-o1': 'diabetica-o1_promptID_001_output_40.xlsx',
                        'diabetica-7B': 'diabetica-7B_promptID_001_output_47.xlsx',
                        'meditron3-8B': 'meditron3-8B_promptID_001_output_39.xlsx',
                        'clinical-chatgpt': 'clinical-chatgpt_promptID_001_output_4.xlsx',
                        'medfound7B': 'medfound7B_promptID_001_output_4.xlsx',
                        'bloom-7b': 'bloom-7b_promptID_001_output_2.xlsx',
                        'llama31': 'llama31_promptID_001_output_8.xlsx',
                        'qwen2-7b': 'qwen2-7b_promptID_001_output_2.xlsx'
                       }

In [ ]:
def counter_extensive_dfs(result_folder, dct_models, letter=True):
    """ 
    Used in experiment 1 to a) keep track of distribution of responses
    b) build dataframe with extensive results.
    
    Variables: 
    result_folder: str, folder containing model's output
    dct_models: str, excel file 
    letter: bool, if True, ESAP question, otherwise, letter removed from option
    
    Returns:
    df_counter: pd.DataFrame, distribution of models responses
    df_all: pd.DataFrame extensive (91 outputs)
    """
    list_df = []
    index = []
    counting_outputs = []

    for model_, file_ in dct_models.items():
        if letter:
            filepath = join(path_to_results, model_.split("-prompt")[0], file_)
        else: 
            filepath = join(path_to_results, "NO_LETTERS", file_)
            
        df_ = pd.read_excel(filepath, header=None, index_col=0)
        df_ = df_[1] # we keep first column, others are empty
        list_df.append(df_)
        
        dct_counter = evaluate_response_distrib(filepath)
        counting_outputs.append([dct_counter[k] for k in sorted(dct_counter.keys())])

    # this is for the counter
    columns_name = sorted(dct_counter.keys())
    
    list_model_name = [m_ if letter else f"{m_}_no_letter" for m_ in dct_models.keys()]

    df_counter = pd.DataFrame(data=counting_outputs, 
                              index=list_model_name,
                              columns=columns_name)

    # this is for the Df with 91 responses
    df_all = pd.concat(list_df, axis=1)
    df_all.columns = list_model_name
    
    return df_counter, df_all

## Load Responses


In [ ]:
df_counter_prompt_A_B, df_all_prompt = counter_extensive_dfs(path_to_results, model_paths_token)
df_counter_no_letter, df_all_no_letter = counter_extensive_dfs(path_to_results, model_paths_no_token, letter=False)

# Combine all prompt variants in one response table.
df_all = pd.concat([df_all_prompt, df_all_no_letter], axis=1)

# Keep only valid ESAP options; all other outputs are treated as missing for agreement.
for col in df_all.columns:
    df_all[col] = df_all[col].where(df_all[col].isin(options), np.nan)

df_all.head()


## Kappa Matrix Helpers


In [ ]:
def cohen_kappa_ci(series_a, series_b, B=10000, ci=95, random_state=42):
    """Return Cohen's kappa and a bootstrap confidence interval for two response series."""
    pair_df = pd.concat([series_a, series_b], axis=1).dropna()
    n = len(pair_df)
    if n == 0:
        return pd.Series({"kappa": np.nan, "ci_low": np.nan, "ci_high": np.nan, "n": 0})

    x = pair_df.iloc[:, 0].to_numpy()
    y = pair_df.iloc[:, 1].to_numpy()
    kappa = cohen_kappa_score(x, y, labels=options)

    rng = np.random.default_rng(random_state)
    bootstrap = np.empty(B)
    for b in range(B):
        idx = rng.integers(0, n, size=n)
        bootstrap[b] = cohen_kappa_score(x[idx], y[idx], labels=options)

    alpha = (100 - ci) / 2
    lo, hi = np.nanpercentile(bootstrap, [alpha, 100 - alpha])
    return pd.Series({"kappa": kappa, "ci_low": lo, "ci_high": hi, "n": n})


def build_kappa_matrices(df, columns, labels=None, B=10000, random_state=42):
    """Build matrices for kappa, CI lower bound, CI upper bound, and pairwise sample size."""
    labels = labels or columns
    kappa = pd.DataFrame(np.nan, index=labels, columns=labels, dtype=float)
    ci_low = kappa.copy()
    ci_high = kappa.copy()
    n_obs = pd.DataFrame(0, index=labels, columns=labels, dtype=int)

    for i, col_i in enumerate(columns):
        for j, col_j in enumerate(columns):
            if i == j:
                pair_n = df[col_i].dropna().shape[0]
                kappa.iloc[i, j] = 1.0
                ci_low.iloc[i, j] = 1.0
                ci_high.iloc[i, j] = 1.0
                n_obs.iloc[i, j] = pair_n
                continue

            stats = cohen_kappa_ci(
                df[col_i],
                df[col_j],
                B=B,
                random_state=random_state + i * len(columns) + j,
            )
            kappa.iloc[i, j] = stats["kappa"]
            ci_low.iloc[i, j] = stats["ci_low"]
            ci_high.iloc[i, j] = stats["ci_high"]
            n_obs.iloc[i, j] = int(stats["n"])

    formatted = kappa.copy().astype(object)
    for row in labels:
        for col in labels:
            formatted.loc[row, col] = f"{kappa.loc[row, col]:.3f} [{ci_low.loc[row, col]:.3f}, {ci_high.loc[row, col]:.3f}]"

    return {
        "formatted": formatted,
        "kappa": kappa,
        "ci_low": ci_low,
        "ci_high": ci_high,
        "n": n_obs,
    }


## Model Groups


In [ ]:
prompt_a_cols = [col for col in model_paths_token if col.endswith("-prompt1")]
prompt_b_cols = [col for col in model_paths_token if col.endswith("-prompt2")]
prompt_a_no_letter_cols = [f"{model}_no_letter" for model in model_paths_no_token]

prompt_a_labels = [col.replace("-prompt1", "") for col in prompt_a_cols]
prompt_b_labels = [col.replace("-prompt2", "") for col in prompt_b_cols]
prompt_a_no_letter_labels = list(model_paths_no_token.keys())

print(f"Prompt A models: {len(prompt_a_cols)}")
print(f"Prompt B models: {len(prompt_b_cols)}")
print(f"Prompt A without letter-token models: {len(prompt_a_no_letter_cols)}")


## Compute Agreement Matrices


In [ ]:
B_BOOTSTRAP = 10000

agreement_prompt_a = build_kappa_matrices(
    df_all,
    prompt_a_cols,
    labels=prompt_a_labels,
    B=B_BOOTSTRAP,
    random_state=101,
)

agreement_prompt_b = build_kappa_matrices(
    df_all,
    prompt_b_cols,
    labels=prompt_b_labels,
    B=B_BOOTSTRAP,
    random_state=202,
)

agreement_prompt_a_no_letter = build_kappa_matrices(
    df_all,
    prompt_a_no_letter_cols,
    labels=prompt_a_no_letter_labels,
    B=B_BOOTSTRAP,
    random_state=303,
)


## Prompt A


In [ ]:
agreement_prompt_a["formatted"]


## Prompt B


In [ ]:
agreement_prompt_b["formatted"]


## Prompt A Without Letter Token


In [ ]:
agreement_prompt_a_no_letter["formatted"]


## Save Tables


In [ ]:
os.makedirs("tables", exist_ok=True)

agreement_prompt_a["formatted"].to_csv("tables/agreement_matrix_prompt_A_kappa_ci.csv")
agreement_prompt_b["formatted"].to_csv("tables/agreement_matrix_prompt_B_kappa_ci.csv")
agreement_prompt_a_no_letter["formatted"].to_csv("tables/agreement_matrix_prompt_A_no_letter_kappa_ci.csv")

agreement_prompt_a["formatted"].to_latex("tables/agreement_matrix_prompt_A_kappa_ci.tex")
agreement_prompt_b["formatted"].to_latex("tables/agreement_matrix_prompt_B_kappa_ci.tex")
agreement_prompt_a_no_letter["formatted"].to_latex("tables/agreement_matrix_prompt_A_no_letter_kappa_ci.tex")
